In [ ]:
# PENDIENTE. CARGAR MODELOS

Para selección iterativa de atributos yo haría un EDA chico, orientado a responder 4 preguntas:

¿La variable se comporta parecido en train y test? → PSI.
¿La variable discrimina CONTINUA vs BAJA+2 en test? → separación de distribuciones.
¿La variable tiene señal univariada? → AUC / diferencia de medias/medianas.
¿El modelo realmente la usa? → importance del modelo, idealmente desde varios modelos.

No haría 20 gráficos por variable. Haría una tabla resumen + 2/3 gráficos globales.

1. PSI train vs test

Asumiendo que X_train, X_test tienen las mismas columnas:

In [ ]:
def psi_num(train, test, bins=10):
    cuts = np.unique(np.quantile(train.dropna(), np.linspace(0, 1, bins + 1)))
    if len(cuts) < 3:
        return np.nan

    a = pd.cut(train, bins=cuts, include_lowest=True).value_counts(normalize=True)
    b = pd.cut(test, bins=cuts, include_lowest=True).value_counts(normalize=True)

    a, b = a.align(b, fill_value=0)
    a = a.clip(1e-6)
    b = b.clip(1e-6)

    return ((a - b) * np.log(a / b)).sum()

Y para todas las variables:

In [ ]:
psi = pd.DataFrame({
    'feature': X_train.columns,
    'PSI': [psi_num(X_train[c], X_test[c]) for c in X_train.columns]
}).sort_values('PSI', ascending=False)

psi.head(20)

Regla práctica para mirar:

< 0.10 → cambio pequeño
0.10–0.25 → mirar
> 0.25 → cambio importante

No lo usaría como criterio automático de eliminación, especialmente en este trabajo. Es una alerta de estabilidad.

2. CONTINUA vs BAJA+2 en TEST

Acá sí me interesa mucho más que PSI. Podés medir AUC univariada para cada atributo:

In [ ]:
from sklearn.metrics import roc_auc_score

y = y_test_true

auc_test = pd.DataFrame({
    'feature': X_test.columns,
    'AUC': [
        roc_auc_score(y, X_test[c]) 
        if X_test[c].nunique() > 1 else np.nan
        for c in X_test.columns
    ]
})

auc_test['AUC_abs'] = (auc_test['AUC'] - 0.5).abs()
auc_test = auc_test.sort_values('AUC_abs', ascending=False)

auc_test.head(20)

Ojo: si una variable tiene relación inversa con baja, AUC puede ser < 0.5. Por eso miro AUC_abs, no solamente AUC.

Esto te da una respuesta muy simple:

"¿Esta variable, por sí sola, contiene información para distinguir BAJA+2 de CONTINUA?"

3. Distribución CONTINUA vs BAJA+2

Para no hacer 100 gráficos, primero elegís las 10–15 variables más interesantes por AUC y ahí sí graficás.

Por ejemplo:

In [ ]:
top_features = auc_test.head(12)['feature'].tolist()

Y un gráfico por variable:

In [ ]:
for col in top_features:
    plt.figure(figsize=(6, 3))
    plt.hist(X_test.loc[y_test_true == 0, col].dropna(), bins=30, alpha=.5, density=True, label='CONTINUA')
    plt.hist(X_test.loc[y_test_true == 1, col].dropna(), bins=30, alpha=.5, density=True, label='BAJA+2')
    plt.title(col)
    plt.legend()
    plt.show()

Esto es mucho más útil para la selección que mirar distribuciones de todas las features.

4. Cruzaría las tres cosas en UNA tabla

Esta sería probablemente la tabla que yo conservaría para la selección iterativa:

In [ ]:
eda_features = (
    psi
    .merge(auc_test[['feature', 'AUC', 'AUC_abs']], on='feature')
)

eda_features.sort_values('AUC_abs', ascending=False).head(30)

Y después agregaría las importancias de tus modelos.

Por ejemplo, si ya tenés un modelo entrenado:

In [ ]:
imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
})

In [ ]:
eda_features = eda_features.merge(imp, on='feature')

5. Feature importance: no me quedaría solamente con LightGBM

Para tu selección iterativa, haría estas tres:

A. LightGBM importance

Ya la tenés:

In [ ]:
model.feature_importances_


B. Permutation importance

Esta es especialmente interesante porque responde:

"¿Cuánto empeora el modelo si rompo esta variable?"

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    model, X_test, y_test_true,
    scoring='roc_auc',
    n_repeats=3,
    random_state=42
)

perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'perm_importance': perm.importances_mean
}).sort_values('perm_importance', ascending=False)

Pero: como tu modelo fue entrenado para BAJA+1 + BAJA+2, mientras que tu negocio mira BAJA+2, esto mide importancia respecto del objetivo de entrenamiento, no necesariamente respecto de la ganancia. Es una distinción importante.

C. Importancia según distintos modelos

Y esto para mí es muy bueno para tu EDA.

Ya que estás entrenando varios modelos, podés guardar: 
feature | abril_b12 | marzo_abril_b12 | all_b12 
con la importancia de cada modelo.
(PENDIENTE)
Después podés ver cuáles aparecen consistentemente importantes.

Mi EDA mínimo para tu trabajo

Yo no haría mucho más que esto:

Tabla principal

feature
PSI_train_test
AUC_continua_baja2
LightGBM_importance
Permutation_importance

Gráficos:

Top 20 AUC CONTINUA vs BAJA+2.
Top 20 PSI train vs test.
Top 20 permutation importance.
Para las features candidatas, distribución CONTINUA vs BAJA+2.

Y después la selección iterativa puede ser muy sencilla:

mantener variables con señal + estabilidad + utilidad para el modelo;
investigar las que tienen PSI alto;
candidatas a sacar las que tienen poca señal, poca importancia y/o son redundantes.

Esto te da un EDA defendible sin convertirlo en otro proyecto entero.